# ProverbGap -- MCQ Generation (Full Benchmark)

## Model Assignment -- from Pilot Study (Appendix A)

| Strategy | Model | Provider | Pilot Score | Reason |
|---|---|---|---|---|
| Zero-Shot (ZS) | qwen-3-235b-a22b-instruct-2507 | Cerebras | 98.7/100 | 1M tokens/day, no RPD cliff, fastest |
| CoT-English | llama-3.3-70b-versatile | Groq | 97.5/100 | Best performer on CoT-EN |
| CoT-CrossLingual | llama-3.1-8b-instant | Groq | 96.6/100 | 14,400 RPD -- never exhausted on CoT-XL |
| Shortcut Evaluator | llama-3.1-8b-instant | Groq | -- | Different family from ZS generator |
| ZS Backup | moonshotai/kimi-k2-instruct | Groq | 97.9/100 | When Cerebras unavailable |

## Why this routing solves the limits problem
- ZS is the highest-volume strategy. Routing it to Cerebras (1M tokens/day, 30 RPM) instead
  of Groq (1,000 RPD/key) avoids the RPD cliff entirely for the most-used strategy.
- CoT-XL caused 503 cascades on large MoE models (Kimi K2). llama3_8b at 14,400 RPD
  handles CoT-XL without ever hitting a service overload.
- Each model is only used where it scored best -- not as a one-size-fits-all generator.

## Tasks
- Task A: Literal meaning MCQ. Generator sees proverb + translation only.
- Task B: Cultural meaning MCQ. Generator sees proverb + translation + cultural context.

## Output
- Per-row CSV checkpoint -- kill and resume at any time
- mcq_results.zip in /kaggle/working after every 100 rows


In [1]:
# Cell 1: Setup
import subprocess, sys, time, os, json, re, random, ast, itertools, threading
import requests, zipfile, glob
from pathlib import Path

def _run(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

print('Installing packages...')
_run('pip install groq pandas openpyxl -q')
import pandas as pd

# -- Groq key loading -------------------------------------------------------
def _load_groq_keys():
    keys = []
    try:
        from kaggle_secrets import UserSecretsClient
        s = UserSecretsClient()
        for name in ['GROQ_API_KEY','GROQ_API_KEY_2','GROQ_API_KEY_3',
                     'GROQ_API_KEY_4','GROQ_API_KEY_5',
                     'GROQ_API_KEY_6','GROQ_API_KEY_7','GROQ_API_KEY_8']:
            try:
                k = s.get_secret(name)
                if k and k.strip(): keys.append(k.strip())
            except: pass
    except: pass
    for name in ['GROQ_API_KEY','GROQ_API_KEY_2','GROQ_API_KEY_3',
                 'GROQ_API_KEY_4','GROQ_API_KEY_5',
                 'GROQ_API_KEY_6','GROQ_API_KEY_7','GROQ_API_KEY_8']:
        k = os.environ.get(name, '')
        if k.strip() and k not in keys: keys.append(k.strip())
    return keys

def _load_single_key(name):
    try:
        from kaggle_secrets import UserSecretsClient
        k = UserSecretsClient().get_secret(name)
        if k and k.strip(): return k.strip()
    except: pass
    return os.environ.get(name, '').strip() or None

GROQ_KEYS    = _load_groq_keys()
CEREBRAS_KEY = _load_single_key('CEREBRAS_API_KEY')
NVIDIA_KEY   = _load_single_key('NVIDIA_API_KEY')

# -- Smart Groq key manager -------------------------------------------------
# Tracks exhausted keys. Skips exhausted keys automatically.
# Raises AllKeysExhausted when all keys have hit their daily RPD limit.

class AllKeysExhausted(Exception):
    pass

_key_exhausted = {}

def _mark_exhausted(key_index):
    _key_exhausted[key_index] = True
    n_left = sum(1 for i in range(len(GROQ_KEYS)) if not _key_exhausted.get(i))
    print(f'    [key {key_index+1}/{len(GROQ_KEYS)} exhausted]  {n_left} key(s) remaining today')
    if n_left == 0:
        raise AllKeysExhausted('All Groq keys have hit their daily RPD limit.')

if GROQ_KEYS:
    from groq import Groq
    _groq_clients = [(i, Groq(api_key=k)) for i, k in enumerate(GROQ_KEYS)]
    _gen_idx  = [0]
    _eval_idx = [0]

    def get_gen_client():
        for _ in range(len(GROQ_KEYS)):
            i, client = _groq_clients[_gen_idx[0] % len(GROQ_KEYS)]
            _gen_idx[0] = (_gen_idx[0] + 1) % len(GROQ_KEYS)
            if not _key_exhausted.get(i):
                return i, client
        raise AllKeysExhausted('All Groq keys exhausted.')

    def get_eval_client():
        for _ in range(len(GROQ_KEYS)):
            i, client = _groq_clients[_eval_idx[0] % len(GROQ_KEYS)]
            _eval_idx[0] = (_eval_idx[0] + 1) % len(GROQ_KEYS)
            if not _key_exhausted.get(i):
                return i, client
        raise AllKeysExhausted('All Groq keys exhausted.')

# -- Keepalive thread -------------------------------------------------------
def _keepalive():
    count = 0
    while True:
        time.sleep(15 * 60); count += 1
        print(f'[keepalive {count}] still running...', flush=True)
threading.Thread(target=_keepalive, daemon=True).start()

OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True)

# -- Status report ----------------------------------------------------------
print()
print('Provider status:')
if GROQ_KEYS:
    print(f'  Groq     : {len(GROQ_KEYS)} key(s)  ->  {len(GROQ_KEYS)*1000:,} RPD (70B CoT-EN + 8B CoT-XL/eval)')
else:
    print('  Groq     : no keys -- add GROQ_API_KEY in Add-ons -> Secrets')

if CEREBRAS_KEY:
    print('  Cerebras : key loaded  ->  1M tokens/day, 30 RPM (ZS overflow)')
else:
    print('  Cerebras : no key (optional -- Groq llama-3.1-8b handles ZS now)')

if NVIDIA_KEY:
    print('  NVIDIA   : key loaded  ->  build.nvidia.com (eval committee + ZS fallback)')
else:
    print('  NVIDIA   : no key -- add NVIDIA_API_KEY for stronger eval committee')
    print('             (free at build.nvidia.com -- sign up with NVIDIA account)')
print()
print('Setup complete.')


Installing packages...

Provider status:
  Groq     : 8 key(s)  ->  8,000 RPD (70B CoT-EN + 8B CoT-XL/eval)
  Cerebras : key loaded  ->  1M tokens/day, 30 RPM (ZS overflow)
  NVIDIA   : key loaded  ->  build.nvidia.com (eval committee + ZS fallback)

Setup complete.


In [2]:
# Cell 2: Config
# -- Model routing (grounded in pilot study, Appendix A) -------------------
# Each strategy is assigned to the model that scored highest on that strategy.
# This avoids exhausting any single provider and prevents 503 cascades.

STRATEGY_MODEL = {
    # strategy_key -> (provider, model_id, delay_seconds)
    'zs':     ('cerebras', 'qwen-3-235b-a22b-instruct-2507', 0.0),
    # Cerebras qwen3-235b: 1M tokens/day, 30 RPM. Pilot WINNER.
    # Pilot: 98.7/100  Parse: 100%
    # Proved in full production run (mcq_results). Best ZS quality.
    # kimi-k2 scored 97.9 but exhausts max_tokens on <think> tags -> parse=0%
    # If Cerebras key missing -> falls back to Groq llama-3.1-8b (96.4)

    'cot_en': ('groq',     'llama-3.3-70b-versatile',         1.0),
    # Groq 70B: 1,000 RPD/key. With 3 keys = 3,000 RPD.
    # Pilot: 97.5/100  Parse: 100%  Distinct: 0.92

    'cot_xl': ('groq',     'llama-3.1-8b-instant',            0.4),
    # Groq 8B: 14,400 RPD/key. Essentially unconstrained.
    # Pilot: 96.6/100  Parse: 100%  Distinct: 0.89
    # CoT-XL caused 503 cascades on Kimi K2 (large MoE + native-lang reasoning).
    # llama3_8b never 503s because it is smaller and on a high-RPD endpoint.
}

ZS_FALLBACK_GROQ_MODEL = 'llama-3.1-8b-instant'
# Pilot: 96.4/100 on ZS  Parse: 100%  Distinct: 0.88
# Groq fallback when Cerebras key is unavailable. Fast + reliable.

# Fallback chain for ZS when Groq keys exhaust:
#   1. NVIDIA nemotron-70b (build.nvidia.com free tier)
#   2. Cerebras qwen-3-235b (slow but highest pilot score 98.7/100)
ZS_FALLBACK_CHAIN = ['nvidia', 'cerebras']

EVAL_MODEL_ID = 'llama-3.1-8b-instant'
NVIDIA_GEN_MODEL  = 'nvidia/nemotron-3-super-120b-a12b'  # ZS fallback (120B, current flagship)
NVIDIA_EVAL_MODEL = 'nvidia/nemotron-3-super-120b-a12b'  # audit committee
# Evaluator: 14,400 RPD/key. Different model family from ZS generator
# (qwen vs llama) -- eliminates self-preference bias (Wang et al. 2024).

TEMPERATURE = 0.85
MAX_TOKENS  = 2000   # fits 5 proverbs x 3 distractors per batch call
SEED        = 42
CEREBRAS_BATCH_SIZE = 5  # Send 5 proverbs per Cerebras call -> 5x fewer API calls
# Cerebras at 30 RPM x 5 proverbs/call = 150 proverbs/min (was 20/min)
# Each batched call takes ~1.5x longer but 5x fewer calls = ~3x net speedup

# Tasks and strategies to run
RUN_TASK_A        = True   # Literal meaning MCQ
RUN_TASK_B        = True   # Cultural meaning MCQ
RUN_ZERO_SHOT     = True
RUN_COT_ENGLISH   = True
RUN_COT_CROSSLING = True

# Yoruba row limit -- set to None for full corpus (~3,500 rows)
YORUBA_LIMIT = 2000

# -- Data paths (Full_Data dataset) ----------------------------------------
_CANDIDATE_DIRS = [
    Path('/kaggle/input/datasets/abrahamsunday123/full-data/Data'),
    Path('/kaggle/input/full-data/Data'),
    Path('/kaggle/input/datasets/abrahamsunday123/proverbeval-clean-data'),
    Path('/kaggle/input/proverbeval-clean-data'),
]
DATA_DIR = next((d for d in _CANDIDATE_DIRS if d.exists()), _CANDIDATE_DIRS[0])

YORUBA_FILE  = DATA_DIR / 'Copy of Copy of Strict_EMNLP_Yoruba_Corpus.xlsx'
ARABIC_FILE  = DATA_DIR / 'Copy of Copy of Strict_EMNLP_MidResource_Corpus_Arabic_hugging_face.csv'
ENGLISH_FILE = DATA_DIR / 'English_Corpus_clean.csv'
FRENCH_FILE  = DATA_DIR / 'Copy of Strict_EMNLP_MidResource_Corpus_French.csv'
SPANISH_FILE = DATA_DIR / 'Copy of Strict_EMNLP_MidResource_Corpus_Spanish.csv'
GERMAN_FILE  = DATA_DIR / 'Copy of Copy of Strict_EMNLP_MidResource_Corpus_German.csv'

print('Model routing (from pilot study):')
for strat, (prov, mid, delay) in STRATEGY_MODEL.items():
    print(f'  {strat:<8} -> {prov:<10} {mid}  (delay={delay}s)')
print(f'  evaluator -> groq       {EVAL_MODEL_ID}')
if not CEREBRAS_KEY:
    print(f'  ZS fallback -> groq     {ZS_FALLBACK_GROQ_MODEL}')
print()
print(f'Tasks: A={RUN_TASK_A}  B={RUN_TASK_B}')
print(f'Strategies: ZS={RUN_ZERO_SHOT}  CoT-EN={RUN_COT_ENGLISH}  CoT-XL={RUN_COT_CROSSLING}')
print()
print('Data paths:')
for label, path in [('Yoruba', YORUBA_FILE), ('Arabic', ARABIC_FILE),
                     ('English', ENGLISH_FILE), ('French', FRENCH_FILE),
                     ('Spanish', SPANISH_FILE), ('German', GERMAN_FILE)]:
    status = 'OK' if path.exists() else 'MISSING'
    print(f'  [{status}]  {label:<10} {path.name}')


Model routing (from pilot study):
  zs       -> cerebras   qwen-3-235b-a22b-instruct-2507  (delay=0.0s)
  cot_en   -> groq       llama-3.3-70b-versatile  (delay=1.0s)
  cot_xl   -> groq       llama-3.1-8b-instant  (delay=0.4s)
  evaluator -> groq       llama-3.1-8b-instant

Tasks: A=True  B=True
Strategies: ZS=True  CoT-EN=True  CoT-XL=True

Data paths:
  [OK]  Yoruba     Copy of Copy of Strict_EMNLP_Yoruba_Corpus.xlsx
  [OK]  Arabic     Copy of Copy of Strict_EMNLP_MidResource_Corpus_Arabic_hugging_face.csv
  [OK]  English    English_Corpus_clean.csv
  [OK]  French     Copy of Strict_EMNLP_MidResource_Corpus_French.csv
  [OK]  Spanish    Copy of Strict_EMNLP_MidResource_Corpus_Spanish.csv
  [OK]  German     Copy of Copy of Strict_EMNLP_MidResource_Corpus_German.csv


In [3]:
# Cell 3: Load Full Corpus -- all 6 languages

def load_yoruba():
    df = pd.read_excel(YORUBA_FILE)
    df = df[df['QA_Flag'] != 'DROP'].reset_index(drop=True)
    df = df.rename(columns={'Source_Text_Yo':'Proverb','Target_Text_En':'Translation'})
    df['Language'] = 'Yoruba'
    df = df[['Sample_ID','Proverb','Translation','Cultural_Context','Language']]
    if YORUBA_LIMIT is not None:
        df = df.iloc[:YORUBA_LIMIT].reset_index(drop=True)
        print(f'  Yoruba: limited to first {YORUBA_LIMIT:,} rows (set YORUBA_LIMIT=None for full corpus)')
    return df

def load_arabic():
    df = pd.read_csv(ARABIC_FILE)
    df = df.rename(columns={'Source_Text_Mid':'Proverb','Target_Text_En':'Translation'})
    df['Language'] = 'Arabic'
    return df[['Sample_ID','Proverb','Translation','Cultural_Context','Language']]

def load_english():
    df = pd.read_csv(ENGLISH_FILE)
    df = df.rename(columns={'Correct_Meaning':'Translation'})
    df['Cultural_Context'] = df['Translation']
    df['Language'] = 'English'
    return df[['Sample_ID','Proverb','Translation','Cultural_Context','Language']]

def _load_mid(path, lang):
    df = pd.read_csv(path) if str(path).endswith('.csv') else pd.read_excel(path)
    df = df.rename(columns={'Source_Text_Mid':'Proverb','Target_Text_En':'Translation'})
    df['Language'] = lang
    if 'Sample_ID' not in df.columns:
        df['Sample_ID'] = [f'{lang[:3].upper()}{str(i+1).zfill(4)}' for i in range(len(df))]
    if 'Cultural_Context' not in df.columns:
        df['Cultural_Context'] = df['Translation']
    return df[['Sample_ID','Proverb','Translation','Cultural_Context','Language']]

df_yo = load_yoruba()
df_ar = load_arabic()
df_en = load_english()
df_fr = _load_mid(FRENCH_FILE,  'French')
df_es = _load_mid(SPANISH_FILE, 'Spanish')
df_de = _load_mid(GERMAN_FILE,  'German')
full_df = pd.concat([df_yo, df_ar, df_en, df_fr, df_es, df_de], ignore_index=True)

LANGUAGES = ['Yoruba','Arabic','English','French','Spanish','German']

print(f'Full corpus: {len(full_df):,} proverbs')
for lang in LANGUAGES:
    n = len(full_df[full_df['Language']==lang])
    print(f'  {lang:<10}: {n:,}')


  Yoruba: limited to first 2,000 rows (set YORUBA_LIMIT=None for full corpus)
Full corpus: 5,444 proverbs
  Yoruba    : 2,000
  Arabic    : 803
  English   : 2,278
  French    : 160
  Spanish   : 62
  German    : 141


In [4]:
# Cell 4: Checkpoint Management + Resume from Previous Run
# ------------------------------------------------------------------
# 1. Back up OLD results from original dataset as mcq2_* (archive)
# 2. Restore partial mcq_* from any CHECKPOINT dataset (for resume)
# 3. Explicitly handle /kaggle/input/datasets/abrahamsunday123/new-mcq
# 4. Cell 7 will skip completed CSVs and resume partial ones
# ------------------------------------------------------------------
import shutil

# Step 1: Archive old results from the original dataset as mcq2_*
_old_dirs = sorted(glob.glob('/kaggle/input/datasets/abrahamsunday123/mcq-resulit-semi'))
if _old_dirs:
    backed = 0
    for _dir in _old_dirs:
        for src_file in sorted(Path(_dir).glob('mcq_*.csv')):
            backup_name = src_file.name.replace('mcq_', 'mcq2_', 1)
            dst = OUT_DIR / backup_name
            if not dst.exists():
                shutil.copy2(str(src_file), str(dst))
                backed += 1
    if backed:
        print(f'Archived {backed} old result(s) as mcq2_* (backup)', flush=True)
    else:
        print('Old results already archived.', flush=True)
else:
    print('No old checkpoint dataset attached.', flush=True)

# Step 2: Explicit checkpoint paths (user-uploaded resume datasets)
_EXPLICIT_CHECKPOINT_DIRS = [
    Path('/kaggle/input/datasets/abrahamsunday123/monday-data-csv'),
    Path('/kaggle/input/new-mcq'),
    Path('/kaggle/input/datasets/abrahamsunday123/last-mcq-run'),
    Path('/kaggle/input/newmcq'),
]

def _restore_csv(src_file, dst_file):
    """Copy or keep the CSV with more rows."""
    global _checkpoint_restored
    if not dst_file.exists():
        shutil.copy2(str(src_file), str(dst_file))
        _checkpoint_restored += 1
        return
    try:
        existing_rows = len(pd.read_csv(dst_file))
        new_rows = len(pd.read_csv(src_file))
        if new_rows > existing_rows:
            shutil.copy2(str(src_file), str(dst_file))
            _checkpoint_restored += 1
    except Exception:
        pass

_checkpoint_restored = 0

for _cp_dir in _EXPLICIT_CHECKPOINT_DIRS:
    if not _cp_dir.exists():
        continue
    print(f'[checkpoint] Found dataset: {_cp_dir}', flush=True)
    # If user uploaded mcq_results.zip, extract it first
    for zf in sorted(_cp_dir.rglob('*.zip')):
        print(f'  [zip] Extracting {zf.name} ...', flush=True)
        try:
            with zipfile.ZipFile(zf, 'r') as z:
                for member in z.namelist():
                    if member.endswith('.csv') and 'mcq' in member:
                        z.extract(member, OUT_DIR)
                        extracted = OUT_DIR / member
                        if extracted.exists() and '/' in member:
                            # flatten any subdirectories
                            flat = OUT_DIR / Path(member).name
                            shutil.move(str(extracted), str(flat))
                            _checkpoint_restored += 1
                        elif extracted.exists():
                            _checkpoint_restored += 1
        except Exception as e:
            print(f'  [zip] Extract error: {e}', flush=True)
    # Copy any loose mcq_*.csv files
    for csv_file in sorted(_cp_dir.rglob('mcq_*.csv')):
        if csv_file.name.startswith('mcq2_'):
            continue
        _restore_csv(csv_file, OUT_DIR / csv_file.name)

# Step 3: Broad fallback search across ALL /kaggle/input directories
_old_dir_set = set(str(d) for d in _old_dirs) if _old_dirs else set()
for _inp in sorted(Path('/kaggle/input').iterdir()):
    _inp_str = str(_inp)
    if _inp_str in _old_dir_set:
        continue
    for csv_file in sorted(_inp.rglob('mcq_*.csv')):
        if csv_file.name.startswith('mcq2_'):
            continue
        _restore_csv(csv_file, OUT_DIR / csv_file.name)

# Step 4: Report status
_existing = sorted(OUT_DIR.glob('mcq_*.csv'))
_existing = [f for f in _existing if not f.name.startswith('mcq2_')]
if _existing:
    print(f'Working dir has {len(_existing)} mcq_* file(s) ready for resume:', flush=True)
    for f in _existing:
        try:
            n = len(pd.read_csv(f))
            print(f'    {f.name}: {n:,} rows', flush=True)
        except Exception:
            print(f'    {f.name}: (unreadable)', flush=True)
else:
    print('Working dir clean -- ready for fresh generation.', flush=True)

print('Resume: partial mcq_* files will be continued automatically.', flush=True)

No old checkpoint dataset attached.
[checkpoint] Found dataset: /kaggle/input/datasets/abrahamsunday123/monday-data-csv
Working dir has 30 mcq_* file(s) ready for resume:
    mcq_a_cot_en_arabic.csv: 803 rows
    mcq_a_cot_en_english.csv: 1,990 rows
    mcq_a_cot_en_french.csv: 160 rows
    mcq_a_cot_en_german.csv: 141 rows
    mcq_a_cot_en_spanish.csv: 62 rows
    mcq_a_cot_en_yoruba.csv: 2,000 rows
    mcq_a_cot_xl_arabic.csv: 803 rows
    mcq_a_cot_xl_english.csv: 2,278 rows
    mcq_a_cot_xl_french.csv: 160 rows
    mcq_a_cot_xl_german.csv: 141 rows
    mcq_a_cot_xl_spanish.csv: 62 rows
    mcq_a_cot_xl_yoruba.csv: 2,000 rows
    mcq_a_zs_arabic.csv: 803 rows
    mcq_a_zs_english.csv: 2,278 rows
    mcq_a_zs_french.csv: 160 rows
    mcq_a_zs_german.csv: 141 rows
    mcq_a_zs_spanish.csv: 62 rows
    mcq_a_zs_yoruba.csv: 2,000 rows
    mcq_b_cot_en_arabic.csv: 803 rows
    mcq_b_cot_en_english.csv: 2,278 rows
    mcq_b_cot_en_french.csv: 160 rows
    mcq_b_cot_en_german.csv: 141 rows

In [5]:
# Cell 5: Prompts, Parser, Metrics, MCQ Assembler, Zip Helper
# (unchanged from proverbgap__4_.ipynb -- all prompts are correct)

SYS_A_ZS = (
    'You are building a proverb benchmark. Generate exactly 3 wrong LITERAL '
    'interpretations of the proverb -- what it could superficially seem to mean '
    'word-by-word, but does not. All 3 must sound plausible as literal readings '
    'but be incorrect. Return ONLY a valid JSON list of 3 strings. '
    'No markdown. No preamble. No explanation.'
)
SYS_A_COT_EN = (
    'You are building a proverb benchmark. '
    'First reason in English about what wrong LITERAL readings a reader might jump to. '
    'Then output ONLY a JSON list of 3 plausible but incorrect literal interpretations. '
    'Format: <reasoning>your thinking</reasoning>["wrong 1","wrong 2","wrong 3"]'
)
SYS_A_COT_XL = (
    'You are building a proverb benchmark. '
    'First reason IN THE SAME LANGUAGE AS THE PROVERB about wrong literal readings. '
    'Then output ONLY a JSON list of 3 plausible but incorrect literal interpretations in English. '
    'Format: <reasoning>native-language reasoning</reasoning>["wrong 1","wrong 2","wrong 3"]'
)
SYS_B_ZS = (
    'You are building a proverb benchmark. Generate exactly 3 wrong CULTURAL '
    'interpretations of a proverb. ALL 3 must sound like genuine wisdom but be '
    'subtly incorrect in their cultural reading. Return ONLY a valid JSON list of 3 strings. '
    'No markdown. No preamble. No explanation.'
)
SYS_B_COT_EN = (
    'You are building a proverb benchmark. '
    'First reason in English about what makes a culturally wrong interpretation '
    'plausible but subtly incorrect. '
    'Then output ONLY a JSON list of 3 wrong cultural interpretations. '
    'Format: <reasoning>your thinking</reasoning>["wrong 1","wrong 2","wrong 3"]'
)
SYS_B_COT_XL = (
    'You are building a proverb benchmark. '
    'First reason IN THE SAME LANGUAGE AS THE PROVERB about wrong cultural readings. '
    'Then output ONLY a JSON list of 3 wrong cultural interpretations in English. '
    'Format: <reasoning>native-language reasoning</reasoning>["wrong 1","wrong 2","wrong 3"]'
)

def make_prompt_a(proverb, translation, lang):
    return (
        f'Proverb ({lang}): {proverb}\n'
        f'English translation: {translation}\n\n'
        'Generate exactly 3 plausible but WRONG literal interpretations.\n'
        'Return ONLY: ["wrong literal 1", "wrong literal 2", "wrong literal 3"]'
    )

def make_prompt_b(proverb, translation, cultural_meaning, lang):
    return (
        f'Proverb ({lang}): {proverb}\n'
        f'English translation: {translation}\n'
        f'Correct cultural meaning: {cultural_meaning}\n\n'
        'Generate exactly 3 HARD wrong cultural interpretations -- '
        'plausible but subtly incorrect.\n'
        'Return ONLY: ["wrong 1", "wrong 2", "wrong 3"]'
    )

def parse_response(raw):
    if not raw: return None
    raw = raw.strip()
    # Strip thinking tags (including unclosed ones from truncated output)
    raw = re.sub(r'<reasoning>.*?</reasoning>', '', raw, flags=re.DOTALL).strip()
    raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    # Handle truncated <think> with no closing tag
    if '<think>' in raw:
        raw = raw.split('</think>')[-1].strip() if '</think>' in raw else ''
    # Strip code fences
    raw = re.sub(r'^```[a-z]*\n?', '', raw, flags=re.MULTILINE)
    raw = re.sub(r'\n?```$',       '', raw, flags=re.MULTILINE)
    raw = raw.strip()
    if not raw: return None

    # Strategy 1: Direct JSON parse
    try:
        p = json.loads(raw)
        if isinstance(p, list):
            d = [str(x).strip() for x in p if str(x).strip()]
            return d[:3] if len(d) >= 3 else None
        if isinstance(p, dict):
            for v in p.values():
                if isinstance(v, list) and len(v) >= 3:
                    return [str(x).strip() for x in v[:3]]
    except json.JSONDecodeError:
        pass

    # Strategy 2: Find JSON array embedded in text (e.g. after preamble)
    arr_match = re.search(r'\[[\s\S]*?\]', raw)
    if arr_match:
        try:
            p = json.loads(arr_match.group())
            if isinstance(p, list):
                d = [str(x).strip() for x in p if str(x).strip()]
                if len(d) >= 3:
                    return d[:3]
        except json.JSONDecodeError:
            pass
        # Try fixing single quotes -> double quotes
        fixed = arr_match.group().replace("'", '"')
        try:
            p = json.loads(fixed)
            if isinstance(p, list):
                d = [str(x).strip() for x in p if str(x).strip()]
                if len(d) >= 3:
                    return d[:3]
        except json.JSONDecodeError:
            pass

    # Strategy 3: Quoted strings (double or single)
    m = re.findall(r'"([^"]{8,300})"', raw)
    if len(m) >= 3:
        return m[:3]
    m = re.findall(r"'([^']{8,300})'", raw)
    if len(m) >= 3:
        return m[:3]

    # Strategy 4: Numbered list (1. xxx  2. xxx  3. xxx)
    m = re.findall(r'^\s*\d+[\.\)]\s*(.{10,300})', raw, re.MULTILINE)
    if len(m) >= 3:
        return [x.strip().strip('"').strip("'") for x in m[:3]]

    # Strategy 5: Dash/bullet list
    m = re.findall(r'^\s*[-•]\s*(.{10,300})', raw, re.MULTILINE)
    if len(m) >= 3:
        return [x.strip().strip('"').strip("'") for x in m[:3]]

    return None

def m_parse(d):    return 1.0 if (d and len(d)>=3) else 0.0
def m_complete(d): return 0.0 if not d or len(d)<3 else sum(1 for x in d if len(str(x).split())>=5)/3
def m_len_ratio(d, ref):
    if not d: return 0.0
    clen = max(len(ref.split()),1)
    r = [len(str(x).split())/clen for x in d if x]
    return round(sum(r)/len(r),2) if r else 0.0
def m_distinct(d):
    if not d or len(d)<2: return 0.0
    def ws(s): return set(str(s).lower().split())
    scores = []
    for i in range(len(d)):
        for j in range(i+1,len(d)):
            a,b = ws(d[i]),ws(d[j]); u = a|b
            if u: scores.append(1-len(a&b)/len(u))
    return round(sum(scores)/len(scores),2) if scores else 0.0
def quality(d, ref):
    return round(m_parse(d)*40 + m_complete(d)*30 + m_distinct(d)*30, 1)
def score_row(d, ref):
    return {'parse':m_parse(d),'complete':m_complete(d),
            'len_r':m_len_ratio(d,ref),'distinct':m_distinct(d),'quality':quality(d,ref)}

def assemble_mcq(correct_meaning, distractors, rng):
    choices = list(distractors[:3]) + [correct_meaning]
    rng.shuffle(choices)
    labels = ['A','B','C','D']
    answer = labels[choices.index(correct_meaning)]
    return {'Choice_A':choices[0],'Choice_B':choices[1],
            'Choice_C':choices[2],'Choice_D':choices[3],'Answer':answer}

def zip_results():
    _all = {}
    for _fp in sorted(glob.glob(str(OUT_DIR / 'mcq_*.csv'))):
        _all[Path(_fp).name] = _fp
    _zpath = OUT_DIR / 'mcq_results.zip'
    with zipfile.ZipFile(_zpath, 'w', zipfile.ZIP_DEFLATED) as _z:
        for _name, _fp in sorted(_all.items()):
            _z.write(_fp, _name)
    print(f'   [zip] mcq_results.zip -- {len(_all)} file(s)')


# -- Batch prompt builders (5 proverbs per Cerebras call) ------------------
# Batching reduces Cerebras API calls by 5x while staying within 30 RPM limit.
# Each batch call takes ~1.5x longer but 5x fewer calls = ~3x net speedup.

def make_batch_prompt_a(batch_rows, lang):
    """Task A batch: literal meaning distractors for up to 5 proverbs."""
    items = []
    for i, row in enumerate(batch_rows, 1):
        items.append(
            f'Item {i}:\n'
            f'  Proverb ({lang}): {row["Proverb"]}\n'
            f'  English translation: {row["Translation"]}'
        )
    joined = '\n\n'.join(items)
    n = len(batch_rows)
    keys = ', '.join(f'"{i}"' for i in range(1, n+1))
    return (
        f'{joined}\n\n'
        f'For each of the {n} items above, generate exactly 3 plausible but WRONG literal '
        f'interpretations.\n'
        f'Return ONLY a JSON object with keys {keys}, each containing a list of 3 strings.\n'
        f'Example: {{"1": ["wrong1","wrong2","wrong3"], "2": ["wrong1","wrong2","wrong3"]}}'
    )

def make_batch_prompt_b(batch_rows, lang):
    """Task B batch: cultural meaning distractors for up to 5 proverbs."""
    items = []
    for i, row in enumerate(batch_rows, 1):
        items.append(
            f'Item {i}:\n'
            f'  Proverb ({lang}): {row["Proverb"]}\n'
            f'  English translation: {row["Translation"]}\n'
            f'  Correct cultural meaning: {row["Cultural_Context"]}'
        )
    joined = '\n\n'.join(items)
    n = len(batch_rows)
    keys = ', '.join(f'"{i}"' for i in range(1, n+1))
    return (
        f'{joined}\n\n'
        f'For each of the {n} items above, generate exactly 3 HARD wrong cultural interpretations -- '
        f'plausible but subtly incorrect.\n'
        f'Return ONLY a JSON object with keys {keys}, each containing a list of 3 strings.\n'
        f'Example: {{"1": ["wrong1","wrong2","wrong3"], "2": ["wrong1","wrong2","wrong3"]}}'
    )

def parse_batch_response(raw, batch_size):
    """
    Parse a batched Cerebras response into a list of distractor sets.
    Returns a list of length batch_size, where each element is either
    a list of 3 strings or None if that item failed to parse.
    """
    if not raw:
        return [None] * batch_size
    raw = raw.strip()
    raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    raw = re.sub(r'^```[a-z]*\n?', '', raw, flags=re.MULTILINE)
    raw = re.sub(r'\n?```$',       '', raw, flags=re.MULTILINE)
    raw = raw.strip()
    results = [None] * batch_size
    try:
        p = json.loads(raw)
        if isinstance(p, dict):
            for i in range(1, batch_size + 1):
                key = str(i)
                if key in p and isinstance(p[key], list) and len(p[key]) >= 3:
                    results[i-1] = [str(x).strip() for x in p[key][:3]]
            return results
    except json.JSONDecodeError:
        pass
    # Fallback: try to extract per-item using regex
    for i in range(1, batch_size + 1):
        pat = rf'["\']?{i}["\']?\s*:\s*(\[.*?\])'
        m = re.search(pat, raw, re.DOTALL)
        if m:
            try:
                d = json.loads(m.group(1))
                if isinstance(d, list) and len(d) >= 3:
                    results[i-1] = [str(x).strip() for x in d[:3]]
            except Exception:
                pass
    return results

print('Prompts, parser, metrics, assembler loaded.')
print('  Task A: literal meaning -- no cultural context to generator')
print('  Task B: cultural meaning -- context provided to generator')


Prompts, parser, metrics, assembler loaded.
  Task A: literal meaning -- no cultural context to generator
  Task B: cultural meaning -- context provided to generator


In [6]:
# Cell 6: API Callers -- Cerebras + Groq with Exponential Backoff
# -----------------------------------------------------------------------
# call_for_strategy(strategy_key, sys_prompt, user_prompt)
#   -> routes to the right provider and model based on STRATEGY_MODEL
#   -> retries on 503/overload with exponential backoff (2,4,8,16,32s)
#   -> falls back from Cerebras to Groq kimi-k2 if Cerebras key missing
# -----------------------------------------------------------------------

MAX_RETRIES  = 5
CB_URL       = 'https://api.cerebras.ai/v1/chat/completions'
OR_URL       = 'https://openrouter.ai/api/v1/chat/completions'

def _backoff_call(call_fn, label=''):
    for attempt in range(MAX_RETRIES):
        try:
            return call_fn()
        except Exception as e:
            err = str(e)
            is_fatal = any(x in err for x in ['404','model_not_found','not found',
                                               'invalid_api_key','401'])
            is_503   = '503' in err or 'unavailable' in err.lower() or 'overloaded' in err.lower()
            is_429   = '429' in err or 'rate_limit' in err.lower() or 'per day' in err.lower()

            if is_fatal:
                print(f'    Fatal error ({label}): {err[:80]}', flush=True)
                return None

            wait = 2 ** (attempt + 1)  # 2, 4, 8, 16, 32
            if is_503:
                if attempt < MAX_RETRIES - 1:
                    print(f'    503 attempt {attempt+1}/{MAX_RETRIES} -- waiting {wait}s...', flush=True)
                    time.sleep(wait)
                else:
                    print(f'    503 failed after {MAX_RETRIES} retries ({label}) -- skipping', flush=True)
            elif is_429:
                if 'cerebras' in label.lower():
                    # Cerebras 30 RPM = 60-second rolling window.
                    # Must wait for full window reset -- short retries all fail.
                    cb_wait = 35
                    print(f'    Cerebras RPM window -- waiting {cb_wait}s for reset...', flush=True)
                    time.sleep(cb_wait)
                else:
                    print(f'    Rate limit ({label}) -- waiting {wait}s...', flush=True)
                    time.sleep(wait)
            else:
                if attempt == 0:
                    print(f'    Error ({label}): {err[:100]}', flush=True)
                if attempt < MAX_RETRIES - 1:
                    time.sleep(min(wait, 10))
    return None

def _call_cerebras(model_id, sys_prompt, user_prompt):
    def _fn():
        r = requests.post(CB_URL, headers={
            'Authorization': f'Bearer {CEREBRAS_KEY}',
            'Content-Type':  'application/json',
        }, json={
            'model': model_id,
            'messages': [{'role':'system','content':sys_prompt},
                         {'role':'user',  'content':user_prompt}],
            'temperature': TEMPERATURE, 'max_tokens': MAX_TOKENS,
        }, timeout=(10, 60))
        if r.status_code == 200:
            return r.json()['choices'][0]['message']['content']
        raise Exception(f'{r.status_code} {r.text[:150]}')
    return _backoff_call(_fn, label=f'cerebras/{model_id}')

def _call_cerebras_batch(model_id, sys_prompt, batch_prompt):
    """
    Batched Cerebras call: one API request covers CEREBRAS_BATCH_SIZE proverbs.
    Same rate limits apply -- but 5x fewer calls needed.
    """
    def _fn():
        r = requests.post(CB_URL, headers={
            'Authorization': f'Bearer {CEREBRAS_KEY}',
            'Content-Type':  'application/json',
        }, json={
            'model': model_id,
            'messages': [{'role':'system','content':sys_prompt},
                         {'role':'user',  'content':batch_prompt}],
            'temperature': TEMPERATURE,
            'max_tokens':  MAX_TOKENS,   # 2000 to fit all 5 items
        }, timeout=(10, 90))
        if r.status_code == 200:
            return r.json()['choices'][0]['message']['content']
        raise Exception(f'{r.status_code} {r.text[:150]}')
    return _backoff_call(_fn, label=f'cerebras_batch/{model_id}')

# -- NVIDIA build.nvidia.com inference API ----------------------------------
NV_URL = 'https://integrate.api.nvidia.com/v1/chat/completions'

def _call_nvidia_gen(model_id, sys_prompt, user_prompt):
    if not NVIDIA_KEY: return None
    def _fn():
        r = requests.post(NV_URL, headers={
            'Authorization': f'Bearer {NVIDIA_KEY}',
            'Content-Type':  'application/json',
        }, json={
            'model': model_id,
            'messages': [{'role':'system','content':sys_prompt},
                         {'role':'user',  'content':user_prompt}],
            'temperature': TEMPERATURE, 'max_tokens': MAX_TOKENS,
        }, timeout=(10, 180))
        if r.status_code == 200:
            return r.json()['choices'][0]['message']['content']
        raise Exception(f'{r.status_code} {r.text[:150]}')
    return _backoff_call(_fn, label=f'nvidia/{model_id}')

def _call_nvidia_eval(prompt):
    """NVIDIA evaluation call for audit committee."""
    if not NVIDIA_KEY: return None
    def _fn():
        r = requests.post(NV_URL, headers={
            'Authorization': f'Bearer {NVIDIA_KEY}',
            'Content-Type':  'application/json',
        }, json={
            'model': NVIDIA_EVAL_MODEL,
            'messages': [{'role':'user','content':prompt}],
            'temperature': 0, 'max_tokens': 10,
        }, timeout=(10, 30))
        if r.status_code == 200:
            return r.json()['choices'][0]['message']['content']
        raise Exception(f'{r.status_code} {r.text[:150]}')
    return _backoff_call(_fn, label=f'nvidia_eval/{NVIDIA_EVAL_MODEL}')

def _call_groq(model_id, sys_prompt, user_prompt):
    if not GROQ_KEYS: return None
    _key_idx, _client = get_gen_client()
    def _fn():
        resp = _client.chat.completions.create(
            model=model_id,
            messages=[{'role':'system','content':sys_prompt},
                      {'role':'user',  'content':user_prompt}],
            temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
        )
        return resp.choices[0].message.content
    try:
        return _backoff_call(_fn, label=f'groq/{model_id}')
    except Exception as e:
        err = str(e)
        if 'rate_limit' in err.lower() or 'per day' in err.lower():
            _mark_exhausted(_key_idx)
            # Retry immediately with next available key
            try:
                _key_idx2, _client2 = get_gen_client()
                def _fn2():
                    resp = _client2.chat.completions.create(
                        model=model_id,
                        messages=[{'role':'system','content':sys_prompt},
                                  {'role':'user',  'content':user_prompt}],
                        temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                    )
                    return resp.choices[0].message.content
                return _backoff_call(_fn2, label=f'groq/{model_id}/key{_key_idx2+1}')
            except AllKeysExhausted:
                raise
        return None

def call_for_strategy(strategy_key, sys_prompt, user_prompt):
    """
    Routes to the correct provider and model.
    Provider chain: groq -> nvidia -> cerebras (for ZS).
    """
    provider, model_id, _ = STRATEGY_MODEL[strategy_key]

    if provider == 'groq':
        return _call_groq(model_id, sys_prompt, user_prompt)

    if provider == 'nvidia':
        return _call_nvidia_gen(model_id, sys_prompt, user_prompt)

    if provider == 'cerebras':
        if CEREBRAS_KEY:
            return _call_cerebras(model_id, sys_prompt, user_prompt)
        else:
            print('    [Cerebras key missing -- using Groq llama-3.1-8b fallback]', flush=True)
            return _call_groq(ZS_FALLBACK_GROQ_MODEL, sys_prompt, user_prompt)

    return None

# -- NVIDIA probe --
if NVIDIA_KEY:
    try:
        _nv_probe = requests.post(NV_URL, headers={
            'Authorization': f'Bearer {NVIDIA_KEY}',
            'Content-Type': 'application/json',
        }, json={
            'model': NVIDIA_GEN_MODEL,
            'messages': [{'role':'user','content':'Say OK'}],
            'temperature': 0, 'max_tokens': 5,
        }, timeout=(10, 30))
        if _nv_probe.status_code == 200:
            print(f'NVIDIA probe: OK', flush=True)
        else:
            print(f'NVIDIA probe: {_nv_probe.status_code} -- {_nv_probe.text[:80]}', flush=True)
    except Exception as _e:
        print(f'NVIDIA probe: FAILED ({str(_e)[:60]})', flush=True)

# -- Quick Cerebras probe: fail fast if API is down --
if CEREBRAS_KEY:
    try:
        _probe = requests.post(CB_URL, headers={
            'Authorization': f'Bearer {CEREBRAS_KEY}',
            'Content-Type': 'application/json',
        }, json={
            'model': 'qwen-3-235b-a22b-instruct-2507',  # hardcoded Cerebras model
            'messages': [{'role':'user','content':'Say OK'}],
            'temperature': 0, 'max_tokens': 5,
        }, timeout=(10, 30))
        if _probe.status_code == 200:
            print(f'Cerebras probe: OK', flush=True)
        else:
            print(f'Cerebras probe: {_probe.status_code} -- {_probe.text[:100]}', flush=True)
            print('WARNING: Cerebras may be down. ZS calls will use backoff/retry.', flush=True)
    except Exception as _e:
        print(f'Cerebras probe FAILED: {str(_e)[:100]}', flush=True)
        print('WARNING: Cerebras may be unreachable. Expect slow ZS generation.', flush=True)

print('API callers ready.', flush=True)
if CEREBRAS_KEY:
    print('  ZS     -> Cerebras qwen-3-235b (pilot winner)       (98.7/100)', flush=True)
else:
    print('  ZS     -> Groq llama-3.1-8b (Cerebras key missing)  (96.4/100)', flush=True)
if NVIDIA_KEY:
    print(f'  ZS overflow -> NVIDIA {NVIDIA_GEN_MODEL}', flush=True)
print('  CoT-EN -> Groq llama-3.3-70b-versatile             (97.5/100)', flush=True)
print('  CoT-XL -> Groq llama-3.1-8b-instant                (96.6/100)', flush=True)

print('  Exponential backoff: 2,4,8,16,32s on 503/overload errors', flush=True)


NVIDIA probe: OK
Cerebras probe: 429 -- {"message":"We're experiencing high traffic right now! Please try again soon.","type":"too_many_requ
API callers ready.
  ZS     -> Cerebras qwen-3-235b (pilot winner)       (98.7/100)
  ZS overflow -> NVIDIA nvidia/nemotron-3-super-120b-a12b
  CoT-EN -> Groq llama-3.3-70b-versatile             (97.5/100)
  CoT-XL -> Groq llama-3.1-8b-instant                (96.6/100)
  Exponential backoff: 2,4,8,16,32s on 503/overload errors


In [7]:
# Cell 7: MCQ Generation -- optimized for throughput & resume
# -----------------------------------------------------------------------
# PARALLELISM STRATEGY (optimized for speed):
#   - Cerebras ZS runs IN PARALLEL with Groq CoT strategies (different APIs)
#   - Groq: 8 concurrent workers per strategy (one per API key)
#   - Cerebras: batched (5/call), sequential (30 RPM limit)
#   - CSV checkpoint: every 5 batches (Cerebras) / every 10 rows (Groq)
#   - zip: every 200 rows
#   - Resume: skips completed CSVs, resumes partial CSVs from last row
# -----------------------------------------------------------------------

import time as _time_mod
from concurrent.futures import ThreadPoolExecutor, as_completed

_gen_start = _time_mod.time()

def _elapsed():
    m, s = divmod(int(_time_mod.time() - _gen_start), 60)
    h, m = divmod(m, 60)
    return f'{h}:{m:02d}:{s:02d}'

# -- Parallel Groq worker -------------------------------------------------
def _groq_worker(args):
    """Process a single proverb using a dedicated Groq key. Thread-safe."""
    key_idx, client, model_id, sys_prompt, user_prompt, strategy_key = args
    try:
        resp = client.chat.completions.create(
            model=model_id,
            messages=[{'role': 'system', 'content': sys_prompt},
                      {'role': 'user',   'content': user_prompt}],
            temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
        )
        return resp.choices[0].message.content
    except Exception as e:
        err = str(e)
        if 'rate_limit' in err.lower() or 'per day' in err.lower():
            return f'__EXHAUSTED__{key_idx}'
        try:
            time.sleep(2)
            resp = client.chat.completions.create(
                model=model_id,
                messages=[{'role': 'system', 'content': sys_prompt},
                          {'role': 'user',   'content': user_prompt}],
                temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
            )
            return resp.choices[0].message.content
        except Exception:
            return None

# How many parallel workers to use for Groq
_N_GROQ_WORKERS = min(len(GROQ_KEYS), 8) if GROQ_KEYS else 0
print(f'Groq parallel workers: {_N_GROQ_WORKERS}', flush=True)

if not GROQ_KEYS and not CEREBRAS_KEY:
    print('No API keys loaded. Add keys in Kaggle Add-ons -> Secrets.', flush=True)
else:
    # Build run matrix
    RUNS = []
    if RUN_TASK_A:
        if RUN_ZERO_SHOT:     RUNS.append(('a','zs',     SYS_A_ZS,     'Task A -- Zero-Shot'))
        if RUN_COT_ENGLISH:   RUNS.append(('a','cot_en', SYS_A_COT_EN, 'Task A -- CoT-English'))
        if RUN_COT_CROSSLING: RUNS.append(('a','cot_xl', SYS_A_COT_XL, 'Task A -- CoT-CrossLingual'))
    if RUN_TASK_B:
        if RUN_ZERO_SHOT:     RUNS.append(('b','zs',     SYS_B_ZS,     'Task B -- Zero-Shot'))
        if RUN_COT_ENGLISH:   RUNS.append(('b','cot_en', SYS_B_COT_EN, 'Task B -- CoT-English'))
        if RUN_COT_CROSSLING: RUNS.append(('b','cot_xl', SYS_B_COT_XL, 'Task B -- CoT-CrossLingual'))

    print(f'Runs: {len(RUNS)} strategies x {len(LANGUAGES)} languages = {len(RUNS)*len(LANGUAGES)} CSV files', flush=True)
    for task, strat, _, label in RUNS:
        prov, mid, delay = STRATEGY_MODEL.get(strat, ('?','?',1.0))
        if not CEREBRAS_KEY and prov == 'cerebras':
            mid = ZS_FALLBACK_GROQ_MODEL; prov = 'groq(fallback)'
        print(f'  {label:<35} -> {prov:<18} {mid}  delay={delay}s', flush=True)

    rng = random.Random(SEED)
    generation_summary = []
    _summary_lock = threading.Lock()
    _all_groq_done = False

    # ======================================================================
    # CORE: process one (task, strategy, language) combination
    # ======================================================================
    def _run_single(task, strategy_key, sys_prompt, run_label, lang, _rng):
        """Generate MCQs for one task/strategy/language. Thread-safe."""
        global _all_groq_done
        lang_df  = full_df[full_df['Language']==lang].reset_index(drop=True)
        csv_name = f'mcq_{task}_{strategy_key}_{lang.lower()}.csv'
        csv_path = OUT_DIR / csv_name

        # Skip if already complete
        if csv_path.exists():
            try:
                _ex = pd.read_csv(csv_path)
                if len(_ex) >= len(lang_df):
                    q = round(_ex['quality'].mean(),1)
                    p = round(_ex['parse'].mean()*100,0)
                    print(f'  [skip] {run_label} {lang:<10} -- {len(_ex):,} rows  parse={p:.0f}%  quality={q}', flush=True)
                    with _summary_lock:
                        generation_summary.append({
                            'task':task,'strategy':strategy_key,'language':lang,
                            'model': STRATEGY_MODEL[strategy_key][1],
                            'n_items':len(_ex),
                            'parse_pct':round(_ex['parse'].mean()*100,1),
                            'complete_pct':round(_ex['complete'].mean()*100,1),
                            'avg_distinct':round(_ex['distinct'].mean(),2),
                            'quality':q,
                        })
                    return
            except Exception:
                pass

        print(f'  {run_label} {lang} ({len(lang_df):,} rows)...', flush=True)
        rows = []
        if csv_path.exists():
            try:
                rows = pd.read_csv(csv_path).to_dict('records')
                print(f'    Resuming from row {len(rows)}', flush=True)
            except Exception:
                rows = []
        done_ids = {str(r['sample_id']) for r in rows}

        # -- English Task B ZS shortcut --------------------------------
        if task == 'b' and strategy_key == 'zs' and lang == 'English':
            src_path = OUT_DIR / f'mcq_a_zs_{lang.lower()}.csv'
            if src_path.exists():
                src_df = pd.read_csv(src_path)
                src_df['task'] = 'b'
                src_df['cultural_context_available'] = True
                src_df.to_csv(csv_path, index=False)
                with _summary_lock:
                    generation_summary.append({
                        'task': 'b', 'strategy': 'zs', 'language': 'English',
                        'model': STRATEGY_MODEL['zs'][1],
                        'n_items': len(src_df),
                        'parse_pct': round(src_df['parse'].mean()*100,1),
                        'complete_pct': round(src_df['complete'].mean()*100,1),
                        'avg_distinct': round(src_df['distinct'].mean(),2),
                        'quality': round(src_df['quality'].mean(),1),
                    })
                print(f'    [copied from Task A ZS -- Cultural==Translation]', flush=True)
                zip_results()
                return
            else:
                print(f'    Task A ZS not done yet -- generating normally', flush=True)

        try:
            provider_for_strat = STRATEGY_MODEL[strategy_key][0]
            if not CEREBRAS_KEY and provider_for_strat == 'cerebras':
                provider_for_strat = 'groq'

            # ==============================================================
            # PATH A: Cerebras batched (sequential -- RPM limited)
            # ==============================================================
            if provider_for_strat == 'cerebras':
                pending = lang_df[~lang_df['Sample_ID'].astype(str).isin(done_ids)].reset_index(drop=True)
                model_id = STRATEGY_MODEL[strategy_key][1]
                n_batches = (len(pending) + CEREBRAS_BATCH_SIZE - 1) // CEREBRAS_BATCH_SIZE
                print(f'    Cerebras: {len(pending)} pending -> {n_batches} batches of {CEREBRAS_BATCH_SIZE}', flush=True)

                for batch_start in range(0, len(pending), CEREBRAS_BATCH_SIZE):
                    batch = pending.iloc[batch_start : batch_start + CEREBRAS_BATCH_SIZE]
                    if task == 'a':
                        batch_prompt = make_batch_prompt_a(batch.to_dict('records'), lang)
                    else:
                        batch_prompt = make_batch_prompt_b(batch.to_dict('records'), lang)

                    batch_raw = _call_cerebras_batch(model_id, sys_prompt, batch_prompt)
                    batch_dists = parse_batch_response(batch_raw, len(batch)) if batch_raw else [None]*len(batch)

                    for local_idx, (_, row) in enumerate(batch.iterrows()):
                        dists       = batch_dists[local_idx]
                        correct_text = str(row['Translation']) if task=='a' else str(row['Cultural_Context'])
                        ref          = correct_text
                        mcq = assemble_mcq(correct_text, dists or ['','',''], _rng) \
                              if dists else {'Choice_A':'','Choice_B':'','Choice_C':'','Choice_D':'','Answer':''}
                        rows.append({
                            'task': task, 'strategy': strategy_key, 'language': lang,
                            'sample_id': str(row['Sample_ID']),
                            'source_proverb': str(row['Proverb']),
                            'proverb_en': str(row['Translation']),
                            'correct_meaning': correct_text,
                            'cultural_context_available': (task == 'b'),
                            **mcq,
                            'distractor_1': dists[0] if dists else '',
                            'distractor_2': dists[1] if dists else '',
                            'distractor_3': dists[2] if dists else '',
                            'raw_response': (batch_raw or '')[:400],
                            **score_row(dists, ref),
                        })

                    # Save every 5 batches (25 items) instead of every batch
                    cur_batch = batch_start // CEREBRAS_BATCH_SIZE + 1
                    if cur_batch % 5 == 0 or batch_start + CEREBRAS_BATCH_SIZE >= len(pending):
                        pd.DataFrame(rows).to_csv(csv_path, index=False)
                    processed = batch_start + len(batch)
                    q = round(sum(r['quality'] for r in rows)/len(rows),1)
                    p = round(sum(r['parse'] for r in rows)/len(rows)*100,0)
                    if cur_batch % 10 == 0 or cur_batch == n_batches:
                        print(f'    batch {cur_batch}/{n_batches}  rows={processed}/{len(pending)}  parse={p:.0f}%  q={q}  [{_elapsed()}]', flush=True)
                    if processed % 200 == 0:
                        zip_results()

                    # No extra sleep -- _backoff_call handles 429s

                if rows:
                    pd.DataFrame(rows).to_csv(csv_path, index=False)
                    df_out = pd.read_csv(csv_path)
                    agg = {
                        'task':task,'strategy':strategy_key,'language':lang,
                        'model': STRATEGY_MODEL[strategy_key][1],
                        'n_items':len(df_out),
                        'parse_pct':round(df_out['parse'].mean()*100,1),
                        'complete_pct':round(df_out['complete'].mean()*100,1),
                        'avg_distinct':round(df_out['distinct'].mean(),2),
                        'quality':round(df_out['quality'].mean(),1),
                    }
                    with _summary_lock:
                        generation_summary.append(agg)
                    print(f'  {run_label} {lang:<10} DONE: {agg["n_items"]:,}  parse={agg["parse_pct"]:.0f}%  distinct={agg["avg_distinct"]:.2f}  quality={agg["quality"]:.1f}/100  [{_elapsed()}]', flush=True)
                    zip_results()
                return

            # ==============================================================
            # PATH B: Groq PARALLEL (up to 8 concurrent API calls)
            # ==============================================================
            pending_rows = [(idx, row) for idx, row in lang_df.iterrows()
                            if str(row['Sample_ID']) not in done_ids]
            print(f'    Groq parallel: {len(pending_rows)} pending, {_N_GROQ_WORKERS} workers', flush=True)

            model_id = STRATEGY_MODEL[strategy_key][1]
            _exhausted_keys = set()
            _save_counter = 0
            gen_delay = STRATEGY_MODEL[strategy_key][2]

            chunk_size = _N_GROQ_WORKERS
            for chunk_start in range(0, len(pending_rows), chunk_size):
                chunk = pending_rows[chunk_start : chunk_start + chunk_size]

                work_items = []
                for i, (idx, row) in enumerate(chunk):
                    key_i = i % len(GROQ_KEYS)
                    if key_i in _exhausted_keys:
                        found = False
                        for k in range(len(GROQ_KEYS)):
                            if k not in _exhausted_keys:
                                key_i = k
                                found = True
                                break
                        if not found:
                            raise AllKeysExhausted('All Groq keys exhausted')

                    _, client = _groq_clients[key_i]
                    if task == 'a':
                        prompt = make_prompt_a(row['Proverb'], row['Translation'], lang)
                    else:
                        prompt = make_prompt_b(row['Proverb'], row['Translation'],
                                               row['Cultural_Context'], lang)
                    work_items.append((key_i, client, model_id, sys_prompt, prompt, strategy_key))

                results = []
                with ThreadPoolExecutor(max_workers=min(len(work_items), _N_GROQ_WORKERS)) as pool:
                    futures = {pool.submit(_groq_worker, wi): i for i, wi in enumerate(work_items)}
                    for future in as_completed(futures):
                        fi = futures[future]
                        results.append((fi, future.result()))

                results.sort(key=lambda x: x[0])
                for ri, (fi, raw) in enumerate(results):
                    idx, row = chunk[fi]

                    if isinstance(raw, str) and raw.startswith('__EXHAUSTED__'):
                        exhausted_key = int(raw.split('__')[-1])
                        _exhausted_keys.add(exhausted_key)
                        _mark_exhausted(exhausted_key)
                        raw = None

                    dists = parse_response(raw) if raw else None
                    correct_text = str(row['Translation']) if task=='a' else str(row['Cultural_Context'])
                    ref = correct_text
                    mcq = assemble_mcq(correct_text, dists or ['','',''], _rng) \
                          if dists else {'Choice_A':'','Choice_B':'','Choice_C':'','Choice_D':'','Answer':''}

                    rows.append({
                        'task': task, 'strategy': strategy_key, 'language': lang,
                        'sample_id': str(row['Sample_ID']),
                        'source_proverb': str(row['Proverb']),
                        'proverb_en': str(row['Translation']),
                        'correct_meaning': correct_text,
                        'cultural_context_available': (task == 'b'),
                        **mcq,
                        'distractor_1': dists[0] if dists else '',
                        'distractor_2': dists[1] if dists else '',
                        'distractor_3': dists[2] if dists else '',
                        'raw_response': (raw or '')[:400] if not (isinstance(raw, str) and raw.startswith('__')) else '',
                        **score_row(dists, ref),
                    })

                _save_counter += len(chunk)

                if _save_counter >= 10 or chunk_start + chunk_size >= len(pending_rows):
                    pd.DataFrame(rows).to_csv(csv_path, index=False)
                    _save_counter = 0

                total_done = chunk_start + len(chunk)
                if total_done % max(chunk_size, 1) == 0 or total_done >= len(pending_rows):
                    q = round(sum(r['quality'] for r in rows)/len(rows),1)
                    p = round(sum(r['parse'] for r in rows)/len(rows)*100,0)
                    print(f'    {total_done:>5}/{len(pending_rows)}  parse={p:.0f}%  q={q}  [{_elapsed()}]', flush=True)

                if total_done % 200 == 0:
                    zip_results()

                time.sleep(gen_delay)

        except AllKeysExhausted:
            if rows:
                pd.DataFrame(rows).to_csv(csv_path, index=False)

            done_ids_now = {str(r['sample_id']) for r in rows}
            remaining = [(idx, row) for idx, row in lang_df.iterrows()
                         if str(row['Sample_ID']) not in done_ids_now]

            if remaining and NVIDIA_KEY:
                print(f'    Groq exhausted -- NVIDIA fallback for {len(remaining)} remaining rows', flush=True)
                for idx, row in remaining:
                    if task == 'a':
                        prompt = make_prompt_a(row['Proverb'], row['Translation'], lang)
                        ref = str(row['Translation'])
                    else:
                        prompt = make_prompt_b(row['Proverb'], row['Translation'],
                                               row['Cultural_Context'], lang)
                        ref = str(row['Cultural_Context'])
                    raw = _call_nvidia_gen(NVIDIA_GEN_MODEL, sys_prompt, prompt)
                    dists = parse_response(raw) if raw else None
                    correct_text = ref
                    mcq = assemble_mcq(correct_text, dists or ['','',''], _rng) \
                          if dists else {'Choice_A':'','Choice_B':'','Choice_C':'','Choice_D':'','Answer':''}
                    rows.append({
                        'task': task, 'strategy': strategy_key, 'language': lang,
                        'sample_id': str(row['Sample_ID']),
                        'source_proverb': str(row['Proverb']),
                        'proverb_en': str(row['Translation']),
                        'correct_meaning': correct_text,
                        'cultural_context_available': (task == 'b'),
                        **mcq,
                        'distractor_1': dists[0] if dists else '',
                        'distractor_2': dists[1] if dists else '',
                        'distractor_3': dists[2] if dists else '',
                        'raw_response': (raw or '')[:400],
                        **score_row(dists, ref),
                    })
                    if len(rows) % 20 == 0:
                        pd.DataFrame(rows).to_csv(csv_path, index=False)
                        print(f'    NVIDIA: {len(rows)}/{len(lang_df)} rows  [{_elapsed()}]', flush=True)
                    time.sleep(1.0)
                pd.DataFrame(rows).to_csv(csv_path, index=False)
                print(f'    NVIDIA fallback complete: {len(rows)} rows', flush=True)

            elif remaining and CEREBRAS_KEY:
                print(f'    NVIDIA unavailable -- Cerebras fallback for {len(remaining)} remaining rows', flush=True)
                model_id = 'qwen-3-235b-a22b-instruct-2507'
                pending_fb = pd.DataFrame([row for _, row in remaining])
                for batch_start in range(0, len(pending_fb), CEREBRAS_BATCH_SIZE):
                    batch = pending_fb.iloc[batch_start:batch_start+CEREBRAS_BATCH_SIZE]
                    if task == 'a':
                        bp = make_batch_prompt_a(batch.to_dict('records'), lang)
                    else:
                        bp = make_batch_prompt_b(batch.to_dict('records'), lang)
                    batch_raw = _call_cerebras_batch(model_id, sys_prompt, bp)
                    batch_dists = parse_batch_response(batch_raw, len(batch)) if batch_raw else [None]*len(batch)
                    for li, (_, row) in enumerate(batch.iterrows()):
                        dists = batch_dists[li]
                        ct = str(row['Translation']) if task=='a' else str(row['Cultural_Context'])
                        mcq = assemble_mcq(ct, dists or ['','',''], _rng) \
                              if dists else {'Choice_A':'','Choice_B':'','Choice_C':'','Choice_D':'','Answer':''}
                        rows.append({
                            'task':task,'strategy':strategy_key,'language':lang,
                            'sample_id':str(row['Sample_ID']),
                            'source_proverb':str(row['Proverb']),
                            'proverb_en':str(row['Translation']),
                            'correct_meaning':ct,
                            'cultural_context_available':(task=='b'),
                            **mcq,
                            'distractor_1':dists[0] if dists else '',
                            'distractor_2':dists[1] if dists else '',
                            'distractor_3':dists[2] if dists else '',
                            'raw_response':(batch_raw or '')[:400],
                            **score_row(dists, ct),
                        })
                    pd.DataFrame(rows).to_csv(csv_path, index=False)
                    done_n = batch_start + len(batch)
                    print(f'    Cerebras fallback: {done_n}/{len(remaining)}  [{_elapsed()}]', flush=True)
                    time.sleep(2.5)
                print(f'    Cerebras fallback complete: {len(rows)} rows', flush=True)
            else:
                print(flush=True)
                print('='*62, flush=True)
                print('ALL PROVIDERS EXHAUSTED', flush=True)
                print('='*62, flush=True)
                print('Progress saved. Re-run to continue.', flush=True)
                print('='*62, flush=True)
                zip_results()
                _all_groq_done = True

        if rows:
            pd.DataFrame(rows).to_csv(csv_path, index=False)
            df_out = pd.read_csv(csv_path)
            model_used = STRATEGY_MODEL[strategy_key][1]
            if not CEREBRAS_KEY and STRATEGY_MODEL[strategy_key][0] == 'cerebras':
                model_used = ZS_FALLBACK_GROQ_MODEL
            agg = {
                'task':task,'strategy':strategy_key,'language':lang,
                'model': model_used,
                'n_items':len(df_out),
                'parse_pct':round(df_out['parse'].mean()*100,1),
                'complete_pct':round(df_out['complete'].mean()*100,1),
                'avg_distinct':round(df_out['dist'].mean() if 'dist' in df_out.columns else df_out['distinct'].mean(),2),
                'quality':round(df_out['quality'].mean(),1),
            }
            with _summary_lock:
                generation_summary.append(agg)
            print(f'  {run_label} {lang:<10} DONE: {agg["n_items"]:,}  parse={agg["parse_pct"]:.0f}%  distinct={agg["avg_distinct"]:.2f}  quality={agg["quality"]:.1f}/100  [{_elapsed()}]', flush=True)
            zip_results()

    # ======================================================================
    # CONCURRENT DISPATCH: Cerebras ZS || Groq CoT run simultaneously
    # ======================================================================
    cerebras_runs = [(t, s, sp, rl) for t, s, sp, rl in RUNS
                     if STRATEGY_MODEL.get(s, ('',))[0] == 'cerebras' and CEREBRAS_KEY]
    groq_runs     = [(t, s, sp, rl) for t, s, sp, rl in RUNS
                     if STRATEGY_MODEL.get(s, ('',))[0] == 'groq']
    fallback_runs = [(t, s, sp, rl) for t, s, sp, rl in RUNS
                     if STRATEGY_MODEL.get(s, ('',))[0] == 'cerebras' and not CEREBRAS_KEY]
    groq_runs.extend(fallback_runs)

    def _run_cerebras_block():
        cb_rng = random.Random(SEED + 1)
        for task, strategy_key, sys_prompt, run_label in cerebras_runs:
            for lang in LANGUAGES:
                _run_single(task, strategy_key, sys_prompt, run_label, lang, cb_rng)

    def _run_groq_block():
        groq_rng = random.Random(SEED + 2)
        for task, strategy_key, sys_prompt, run_label in groq_runs:
            if _all_groq_done and STRATEGY_MODEL[strategy_key][0] == 'groq':
                print(f'  [skip -- all Groq keys exhausted] {run_label}', flush=True)
                continue
            for lang in LANGUAGES:
                _run_single(task, strategy_key, sys_prompt, run_label, lang, groq_rng)

    if cerebras_runs and groq_runs:
        print(f'\n*** CONCURRENT MODE: {len(cerebras_runs)} Cerebras ZS runs || {len(groq_runs)} Groq CoT runs ***', flush=True)
        print(f'    Cerebras and Groq use independent APIs -- running simultaneously.', flush=True)
        cb_thread = threading.Thread(target=_run_cerebras_block, name='cerebras-zs')
        cb_thread.start()
        _run_groq_block()
        cb_thread.join()
        print(f'\n*** Both Cerebras and Groq blocks complete. [{_elapsed()}] ***', flush=True)
    elif cerebras_runs:
        print(f'\n*** Cerebras-only mode: {len(cerebras_runs)} runs ***', flush=True)
        _run_cerebras_block()
    elif groq_runs:
        print(f'\n*** Groq-only mode: {len(groq_runs)} runs ***', flush=True)
        _run_groq_block()
    else:
        print('No runs to execute.', flush=True)

    if generation_summary:
        sum_df = pd.DataFrame(generation_summary)
        sum_df.to_csv(OUT_DIR/'mcq_generation_summary.csv', index=False)
        zip_results()
        print(flush=True)
        print('='*82, flush=True)
        print('MCQ GENERATION SUMMARY', flush=True)
        print('='*82, flush=True)
        print(f"  {'Task':<6} {'Strategy':<10} {'Language':<10} {'Model':<35} {'Items':>6} {'Parse%':>7} {'Quality':>9}", flush=True)
        print('-'*82, flush=True)
        for _, r in sum_df.iterrows():
            mid_short = r['model'].split('/')[-1][:30]
            print(f"  {r['task'].upper():<6} {r['strategy']:<10} {r['language']:<10} "
                  f"{mid_short:<35} {r['n_items']:>6,} "
                  f"{r['parse_pct']:>6.0f}% {r['quality']:>8.1f}/100", flush=True)
        print('='*82, flush=True)
        print(f'\nTotal generation time: {_elapsed()}', flush=True)

Groq parallel workers: 8
Runs: 6 strategies x 6 languages = 36 CSV files
  Task A -- Zero-Shot                 -> cerebras           qwen-3-235b-a22b-instruct-2507  delay=0.0s
  Task A -- CoT-English               -> groq               llama-3.3-70b-versatile  delay=1.0s
  Task A -- CoT-CrossLingual          -> groq               llama-3.1-8b-instant  delay=0.4s
  Task B -- Zero-Shot                 -> cerebras           qwen-3-235b-a22b-instruct-2507  delay=0.0s
  Task B -- CoT-English               -> groq               llama-3.3-70b-versatile  delay=1.0s
  Task B -- CoT-CrossLingual          -> groq               llama-3.1-8b-instant  delay=0.4s

*** CONCURRENT MODE: 2 Cerebras ZS runs || 4 Groq CoT runs ***
    Cerebras and Groq use independent APIs -- running simultaneously.
  [skip] Task A -- Zero-Shot Yoruba     -- 2,000 rows  parse=82%  quality=78.4
  [skip] Task A -- CoT-English Yoruba     -- 2,000 rows  parse=98%  quality=92.5
  [skip] Task A -- Zero-Shot Arabic     -- 803 ro

In [8]:
# Cell 8: Shortcut Audit -- PARALLEL (Wang et al. 2024)
# -----------------------------------------------------------------------
# Evaluator answers blind -- no correct answer shown.
# Target: close to 25% (random chance for 4 choices).
# Above 45% = distractors have surface shortcuts -- review those items.
# OPTIMIZED: 8 parallel Groq workers (same as generation), ~8x faster.
# -----------------------------------------------------------------------

if not GROQ_KEYS:
    print('No Groq keys -- skipping audit.')
else:
    AUDIT_SYS = (
        'You are answering a multiple-choice question about a proverb. '
        'Select the option that best matches the question. '
        'Reply with ONLY the letter: A, B, C, or D.'
    )
    AUDIT_QUESTION = {
        'a': 'What does this proverb most likely mean literally?',
        'b': 'Which option best captures the cultural meaning of this proverb?',
    }

    def audit_prompt(task, proverb, translation, ca, cb, cc, cd):
        return (
            f'Proverb: {proverb}\n'
            f'English: {translation}\n\n'
            f'{AUDIT_QUESTION[task]}\n'
            f'A. {ca}\nB. {cb}\nC. {cc}\nD. {cd}\n\n'
            'Answer (A/B/C/D only):'
        )

    def parse_audit(raw):
        if not raw: return None
        raw = raw.strip().upper()
        for ch in ['A','B','C','D']:
            if raw.startswith(ch): return ch
        m = re.search(r'\b([ABCD])\b', raw)
        return m.group(1) if m else None

    # -- Parallel audit worker (Groq) --
    def _audit_worker_groq(args):
        """Evaluate one MCQ item using a dedicated Groq key. Thread-safe."""
        key_idx, client, prompt = args
        try:
            resp = client.chat.completions.create(
                model=EVAL_MODEL_ID,
                messages=[{'role':'system','content':AUDIT_SYS},
                          {'role':'user',  'content':prompt}],
                temperature=0.0, max_tokens=5,
            )
            return parse_audit(resp.choices[0].message.content)
        except Exception as e:
            err = str(e)
            if 'rate_limit' in err.lower() or 'per day' in err.lower():
                return f'__EXHAUSTED__{key_idx}'
            try:
                time.sleep(2)
                resp = client.chat.completions.create(
                    model=EVAL_MODEL_ID,
                    messages=[{'role':'system','content':AUDIT_SYS},
                              {'role':'user',  'content':prompt}],
                    temperature=0.0, max_tokens=5,
                )
                return parse_audit(resp.choices[0].message.content)
            except Exception:
                return None

    _nvidia_eval_available = bool(NVIDIA_KEY)
    _n_audit_workers = min(len(GROQ_KEYS), 8) if GROQ_KEYS else 0
    if _nvidia_eval_available:
        print(f'Running shortcut audit: 2-model committee (Groq 8B [{_n_audit_workers} workers] + NVIDIA Nemotron)')
    else:
        print(f'Running shortcut audit: Groq 8B [{_n_audit_workers} parallel workers]')
    print('  Evaluators answer blind -- no correct answer shown.', flush=True)
    print('  Target: ~25% (random chance). Above 45% = surface shortcuts.', flush=True)
    audit_results = []

    for task in ['a','b']:
        for lang in LANGUAGES:
            zs_path    = OUT_DIR / f'mcq_{task}_zs_{lang.lower()}.csv'
            audit_path = OUT_DIR / f'mcq_audit_{task}_zs_{lang.lower()}.csv'

            if not zs_path.exists():
                print(f'  Task {task.upper()} {lang}: zs CSV not found -- run Cell 7 first.')
                continue

            zs_df = pd.read_csv(zs_path)
            if audit_path.exists():
                try:
                    _a = pd.read_csv(audit_path)
                    if len(_a) >= len(zs_df):
                        acc  = round(_a['eval_correct'].mean()*100,1)
                        flag = '  REVIEW' if acc > 45 else ''
                        print(f'  [skip] Task {task.upper()} {lang:<10} accuracy={acc:.1f}%{flag}')
                        audit_results.append({'task':task,'language':lang,'n':len(_a),'accuracy':acc})
                        continue
                except Exception: pass

            print(f'  Task {task.upper()} {lang} ({len(zs_df):,} items)...')
            a_rows, done_ids = [], set()
            if audit_path.exists():
                try:
                    _ap = pd.read_csv(audit_path)
                    a_rows   = _ap.to_dict('records')
                    done_ids = {str(r['sample_id']) for r in a_rows}
                except Exception: pass

            pending = [(_, row) for _, row in zs_df.iterrows()
                       if str(row['sample_id']) not in done_ids]

            if not pending:
                continue

            # -- PARALLEL Groq audit (8 workers) --
            _exhausted_audit = set()
            _audit_broken = False
            chunk_size = _n_audit_workers

            for chunk_start in range(0, len(pending), chunk_size):
                if _audit_broken:
                    break
                chunk = pending[chunk_start : chunk_start + chunk_size]

                work_items = []
                for i, (_, row) in enumerate(chunk):
                    key_i = i % len(GROQ_KEYS)
                    if key_i in _exhausted_audit:
                        found = False
                        for k in range(len(GROQ_KEYS)):
                            if k not in _exhausted_audit:
                                key_i = k
                                found = True
                                break
                        if not found:
                            print('    All evaluator keys exhausted -- audit checkpointed.', flush=True)
                            _audit_broken = True
                            break
                    _, client = _groq_clients[key_i]
                    prompt = audit_prompt(
                        task, row['source_proverb'], row['proverb_en'],
                        row['Choice_A'], row['Choice_B'], row['Choice_C'], row['Choice_D']
                    )
                    work_items.append((key_i, client, prompt))

                if _audit_broken or not work_items:
                    break

                results = []
                with ThreadPoolExecutor(max_workers=min(len(work_items), _n_audit_workers)) as pool:
                    futures = {pool.submit(_audit_worker_groq, wi): i for i, wi in enumerate(work_items)}
                    for future in as_completed(futures):
                        fi = futures[future]
                        results.append((fi, future.result()))

                results.sort(key=lambda x: x[0])
                for ri, (fi, eval_answer) in enumerate(results):
                    _, row = chunk[fi]

                    if isinstance(eval_answer, str) and eval_answer.startswith('__EXHAUSTED__'):
                        exhausted_key = int(eval_answer.split('__')[-1])
                        _exhausted_audit.add(exhausted_key)
                        eval_answer = None

                    eval_answer_nv = None
                    if _nvidia_eval_available and eval_answer is not None:
                        try:
                            prompt = audit_prompt(
                                task, row['source_proverb'], row['proverb_en'],
                                row['Choice_A'], row['Choice_B'], row['Choice_C'], row['Choice_D']
                            )
                            nv_raw = _call_nvidia_eval(AUDIT_SYS + '\n\n' + prompt)
                            eval_answer_nv = parse_audit(nv_raw)
                        except Exception:
                            pass
                        if eval_answer_nv is not None and eval_answer != eval_answer_nv:
                            eval_answer = None

                    a_rows.append({
                        'task':task,'sample_id':row['sample_id'],'language':lang,
                        'correct_pos':row['Answer'],
                        'eval_groq':eval_answer,
                        'eval_nvidia':eval_answer_nv,
                        'eval_answer':eval_answer,
                        'eval_correct':int(eval_answer==row['Answer']) if eval_answer else 0,
                    })

                pd.DataFrame(a_rows).to_csv(audit_path, index=False)

                total_done = chunk_start + len(chunk)
                if total_done % (chunk_size * 10) == 0 or total_done >= len(pending):
                    acc_so_far = round(sum(r['eval_correct'] for r in a_rows)/max(len(a_rows),1)*100,1)
                    print(f'    {total_done:>5}/{len(pending)}  acc={acc_so_far:.1f}%  [{_elapsed()}]', flush=True)

                time.sleep(0.1)

            if a_rows:
                a_df  = pd.read_csv(audit_path)
                acc   = round(a_df['eval_correct'].mean()*100,1)
                flag  = '  REVIEW DISTRACTORS' if acc > 45 else ''
                audit_results.append({'task':task,'language':lang,'n':len(a_df),'accuracy':acc})
                print(f'  Task {task.upper()} {lang:<10} accuracy={acc:.1f}%  '
                      f'(baseline=25%)  n={len(a_df):,}{flag}')
                zip_results()

    if audit_results:
        print()
        print('='*62)
        print('SHORTCUT AUDIT -- Wang et al. 2024')
        print('='*62)
        print(f"  {'Task':<6} {'Language':<12} {'N':>6} {'Acc%':>7} {'vs 25%':>8}")
        print('-'*62)
        for r in audit_results:
            delta = r['accuracy'] - 25.0
            flag  = ' REVIEW' if delta > 20 else ''
            print(f"  {r['task'].upper():<6} {r['language']:<12} {r['n']:>6,} "
                  f"{r['accuracy']:>6.1f}%  {delta:>+6.1f}pp{flag}")
        print('='*62)
        print('  Target: close to 25%. Above 45% = surface shortcuts.')

Running shortcut audit: 2-model committee (Groq 8B [8 workers] + NVIDIA Nemotron)
  Evaluators answer blind -- no correct answer shown.
  Target: ~25% (random chance). Above 45% = surface shortcuts.
  Task A Yoruba (2,000 items)...
       80/2000  acc=63.7%  [0:39:15]
      160/2000  acc=66.2%  [0:41:53]
      240/2000  acc=70.8%  [0:44:39]
[keepalive 3] still running...
      320/2000  acc=68.1%  [0:47:38]
      400/2000  acc=68.8%  [0:53:55]
[keepalive 4] still running...
    Error (nvidia_eval/nvidia/nemotron-3-super-120b-a12b): HTTPSConnectionPool(host='integrate.api.nvidia.com', port=443): Read timed out. (read timeout=30)
      480/2000  acc=68.1%  [1:01:08]
      560/2000  acc=68.8%  [1:09:26]
[keepalive 5] still running...
      640/2000  acc=67.7%  [1:19:45]
    Error (nvidia_eval/nvidia/nemotron-3-super-120b-a12b): HTTPSConnectionPool(host='integrate.api.nvidia.com', port=443): Read timed out. (read timeout=30)
      720/2000  acc=65.0%  [1:28:06]
    All evaluator keys exhau

In [9]:
# Cell 9: Final Summary + Paper Stats + Final ZIP

all_gen = []
for f in sorted(glob.glob(str(OUT_DIR / 'mcq_*.csv'))):
    if 'audit' in f or 'summary' in f: continue
    try:
        _df = pd.read_csv(f); _df['_file'] = Path(f).name
        all_gen.append(_df)
    except Exception: pass

if not all_gen:
    print('No generation CSVs found. Run Cell 7 first.')
else:
    gen_df = pd.concat(all_gen, ignore_index=True)

    print('='*82)
    print('PROVERBGAP MCQ -- FULL RESULTS')
    print('='*82)
    print(f'  Total items : {len(gen_df):,}')
    print()

    for task in sorted(gen_df['task'].unique()):
        task_label = 'Task A (Literal)' if task=='a' else 'Task B (Cultural)'
        t_df = gen_df[gen_df['task']==task]
        for strat in sorted(t_df['strategy'].unique()):
            s_df = t_df[t_df['strategy']==strat]
            print(f'  {task_label}  --  {strat}')
            print(f"  {'Language':<12} {'Items':>7} {'Parse%':>8} {'Dist':>6} {'Quality':>9}")
            print(f"  {'-'*46}")
            for lang in LANGUAGES:
                l_df = s_df[s_df['language']==lang]
                if not len(l_df): continue
                print(f"  {lang:<12} {len(l_df):>7,} "
                      f"{l_df['parse'].mean()*100:>7.0f}% "
                      f"{l_df['distinct'].mean():>6.2f} "
                      f"{l_df['quality'].mean():>8.1f}/100")
            print()

    zs_df     = gen_df[gen_df['strategy']=='zs'] if 'zs' in gen_df['strategy'].values else gen_df
    n_total   = len(zs_df)
    parse_avg = round(zs_df['parse'].mean()*100, 1)
    qual_avg  = round(zs_df['quality'].mean(), 1)

    print('='*82)
    print('PAPER STATEMENT -- Section 3 Dataset Construction')
    print('='*82)
    print(
        'MCQ items were generated using a strategy-specific model routing scheme '
        'grounded in a formal 7-model pilot study (Appendix A). Zero-shot items '
        'were generated by Qwen3-235B (Cerebras, 98.7/100 pilot score). '
        'Chain-of-thought English items used LLaMA 3.3-70B (Groq, 97.5/100). '
        'Cross-lingual chain-of-thought items used LLaMA 3.1-8B (Groq, 96.6/100), '
        'which avoids the 503 service overloads observed on large MoE models '
        'when generating native-language reasoning (pilot study observation). '
        'Two task types were constructed: Task A (literal meaning) and Task B '
        '(cultural meaning). '
        f'A total of {n_total:,} zero-shot MCQ items were generated across '
        '6 languages (Yoruba, Arabic, English, French, Spanish, German). '
        f'The zero-shot strategy achieved a mean parse rate of {parse_avg:.0f}% '
        f'and mean quality score of {qual_avg:.1f}/100. '
        'Correct answers were assigned to random positions (A-D, seed=42) to '
        'control for answer-order bias (Azime et al., NAACL 2025). '
        'Distractor quality was audited using LLaMA 3.1-8B as a blind evaluator '
        '(Wang et al., 2024), a distinct model family from the ZS generator, '
        'which eliminates self-preference bias in the quality audit loop.'
    )
    print('='*82)

    zip_results()
    zip_path = OUT_DIR / 'mcq_results.zip'
    size_kb  = zip_path.stat().st_size / 1024
    print(f'\nFinal zip: {zip_path}  ({size_kb:.0f} KB)')
    print('Download from the Kaggle output panel on the right.')


PROVERBGAP MCQ -- FULL RESULTS
  Total items : 32,376

  Task A (Literal)  --  cot_en
  Language       Items   Parse%   Dist   Quality
  ----------------------------------------------
  Yoruba         2,000      98%   0.80     92.5/100
  Arabic           803      98%   0.77     90.4/100
  English        1,990     100%   0.84     94.2/100
  French           160     100%   0.89     95.6/100
  Spanish           62     100%   0.87     95.5/100
  German           141     100%   0.88     95.2/100

  Task A (Literal)  --  cot_xl
  Language       Items   Parse%   Dist   Quality
  ----------------------------------------------
  Yoruba         2,000      94%   0.73     86.1/100
  Arabic           803      95%   0.74     86.3/100
  English        2,278      91%   0.76     85.3/100
  French           160      95%   0.75     86.6/100
  Spanish           62      92%   0.71     82.4/100
  German           141      94%   0.72     83.9/100

  Task A (Literal)  --  zs
  Language       Items   Parse%   